# Split PDF

In [ ]:
#@title Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Split
file_path = "" #@param {type:"string"}
ranges = "" #@param {type:"string"}

import os
if not file_path:
    raise SystemExit("Enter a file path.")
if not os.path.isfile(file_path):
    raise SystemExit(f"Not a file: {file_path}")
!pip install -q pypdf
from pypdf import PdfReader, PdfWriter
reader = PdfReader(file_path)
n = len(reader.pages)
# Empty means one file per page; otherwise e.g. "1-3, 5, 8-10".
try:
    spans = [(int(a), int(b or a)) for a, _, b in (r.strip().partition("-") for r in ranges.split(",") if r.strip())]
except ValueError:
    raise SystemExit("Use ranges like 1-3, 5, 8-10.")
spans = spans or [(i, i) for i in range(1, n + 1)]
for a, b in spans:
    if not 1 <= a <= b <= n:
        raise SystemExit(f"Range {a}-{b} is outside 1-{n}.")
base = os.path.splitext(file_path)[0]
for a, b in spans:
    writer = PdfWriter()
    for i in range(a - 1, b):
        writer.add_page(reader.pages[i])
    output = f"{base}-p{a}.pdf" if a == b else f"{base}-p{a}-{b}.pdf"
    writer.write(output)
    print("✅", output)
